# Step 6 — SER Dataset Quality Check

Verifies the downloaded `data/ser/` and `data/ser_cropped/` datasets:
- No corrupt JPEG files
- Visual grids (16 images per class, full + cropped side-by-side)
- Class balance and split size table
- IR/night image flag

In [ ]:
import csv
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
REPO_ROOT = Path("../../..").resolve()  # adjust if notebook is moved
DATA_DIR = REPO_ROOT / "data"
SER_DIR = DATA_DIR / "ser"
CROPPED_DIR = DATA_DIR / "ser_cropped"
MANIFEST_PATH = Path("ser_manifest.json")

GRID_COLS = 4
GRID_ROWS = 4  # 16 images per class
SEED = 42
random.seed(SEED)

print(f"SER dir:     {SER_DIR}")
print(f"Cropped dir: {CROPPED_DIR}")
print(f"Exists: ser={SER_DIR.exists()}, cropped={CROPPED_DIR.exists()}")

## 1 — Check for corrupt JPEG files

In [ ]:
def check_corrupt(root: Path) -> list[Path]:
    """Try to open every JPEG under root. Return list of corrupt files."""
    corrupt = []
    all_jpgs = list(root.glob("**/*.jpg"))
    print(f"Checking {len(all_jpgs)} files in {root.name}...")
    for p in all_jpgs:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception as e:
            corrupt.append((p, str(e)))
    return corrupt


corrupt_ser = check_corrupt(SER_DIR)
corrupt_cropped = check_corrupt(CROPPED_DIR)

print(f"\nCorrupt in ser/:        {len(corrupt_ser)}")
print(f"Corrupt in ser_cropped: {len(corrupt_cropped)}")
for p, err in corrupt_ser[:5]:
    print(f"  {p.name}: {err}")

## 2 — Split size and class balance

In [ ]:
def count_images(root: Path) -> dict:
    """Count images per split/label."""
    counts = defaultdict(Counter)  # split -> label -> count
    for jpg in root.glob("**/*.jpg"):
        parts = jpg.relative_to(root).parts
        if len(parts) >= 2:
            split, label = parts[0], parts[1]
            counts[split][label] += 1
    return counts


counts = count_images(SER_DIR)
all_labels = sorted({lbl for split_counts in counts.values() for lbl in split_counts})
splits = ["train", "val", "test"]

print(f"{'label':<25} {'train':>8} {'val':>8} {'test':>8} {'total':>8}")
print("-" * 56)
for label in all_labels:
    tr = counts["train"][label]
    va = counts["val"][label]
    te = counts["test"][label]
    print(f"{label:<25} {tr:>8} {va:>8} {te:>8} {tr + va + te:>8}")

all_totals = {split: sum(counts[split].values()) for split in splits}
grand = sum(all_totals.values())
print(
    f"{'TOTAL':<25} {all_totals.get('train', 0):>8} {all_totals.get('val', 0):>8} {all_totals.get('test', 0):>8} {grand:>8}"
)

In [ ]:
# Class balance check
total_per_label = {lbl: sum(counts[s][lbl] for s in splits) for lbl in all_labels}
max_count = max(total_per_label.values())
min_count = min(total_per_label.values())
print(
    f"Max/min class ratio: {max_count / min_count:.1f}x  (max={max_count}, min={min_count})"
)

fig, ax = plt.subplots(figsize=(max(6, len(all_labels) * 1.2), 4))
ax.bar(
    total_per_label.keys(),
    total_per_label.values(),
    color="steelblue",
    edgecolor="white",
)
ax.set_title("Total images per class (ser/)")
ax.set_ylabel("Images")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

## 3 — Visual grid: full-frame images

In [ ]:
def show_grid(
    root: Path, label: str, n_cols: int = 4, n_rows: int = 4, title_prefix: str = ""
):
    """Display a random grid of n_colsxn_rows images for one label."""
    all_imgs = []
    for split in ["train", "val", "test"]:
        all_imgs.extend((root / split / label).glob("*.jpg"))
    if not all_imgs:
        print(f"  No images found for label '{label}' in {root.name}")
        return
    sample = random.sample(all_imgs, min(n_cols * n_rows, len(all_imgs)))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
    fig.suptitle(f"{title_prefix}{label}  ({root.name})", fontsize=12, y=1.01)
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(sample):
            with Image.open(sample[i]) as im:
                ax.imshow(np.array(im))
    plt.tight_layout()
    plt.show()


for label in all_labels:
    show_grid(SER_DIR, label, n_cols=GRID_COLS, n_rows=GRID_ROWS)

## 4 — Visual grid: cropped images

In [ ]:
animal_labels = [lbl for lbl in all_labels if lbl != "empty"]
for label in animal_labels:
    show_grid(
        CROPPED_DIR,
        label,
        n_cols=GRID_COLS,
        n_rows=GRID_ROWS,
        title_prefix="[cropped] ",
    )

## 5 — Full vs. cropped side-by-side comparison

In [ ]:
def show_side_by_side(label: str, n: int = 4):
    """Show n images from label with full frame (top) and crop (bottom)."""
    full_imgs = []
    for split in ["train", "val", "test"]:
        full_imgs.extend((SER_DIR / split / label).glob("*.jpg"))
    if not full_imgs:
        return

    sample_full = random.sample(full_imgs, min(n, len(full_imgs)))
    fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
    fig.suptitle(f"{label} — full (top) vs cropped (bottom)", fontsize=11)
    for col, p_full in enumerate(sample_full):
        # Find corresponding crop
        image_id = p_full.stem
        p_crop = None
        for split in ["train", "val", "test"]:
            candidate = CROPPED_DIR / split / label / f"{image_id}.jpg"
            if candidate.exists():
                p_crop = candidate
                break

        axes[0, col].axis("off")
        axes[1, col].axis("off")
        with Image.open(p_full) as im:
            axes[0, col].imshow(np.array(im))
        axes[0, col].set_title(f"{im.width}x{im.height}", fontsize=8)
        if p_crop:
            with Image.open(p_crop) as im:
                axes[1, col].imshow(np.array(im))
            axes[1, col].set_title(f"{im.width}x{im.height}", fontsize=8)
        else:
            axes[1, col].text(
                0.5,
                0.5,
                "no crop",
                ha="center",
                va="center",
                transform=axes[1, col].transAxes,
            )
    plt.tight_layout()
    plt.show()


for label in animal_labels[:4]:  # show first 4 animal classes
    show_side_by_side(label)

## 6 — IR / night image detection

IR images tend to have very low colour saturation (nearly greyscale). We flag
images where the mean absolute deviation of per-channel mean is below a
threshold.

In [ ]:
IR_SATURATION_THRESHOLD = 5.0  # mean abs deviation of RGB channels
IR_SAMPLE = 200  # images to sample for speed

all_jpgs = list(SER_DIR.glob("**/*.jpg"))
sample_jpgs = random.sample(all_jpgs, min(IR_SAMPLE, len(all_jpgs)))

ir_count = 0
for p in sample_jpgs:
    with Image.open(p) as im:
        arr = np.array(im.convert("RGB"), dtype=float)
        channel_means = arr.mean(axis=(0, 1))  # R, G, B means
        saturation = float(np.abs(channel_means - channel_means.mean()).mean())
        if saturation < IR_SATURATION_THRESHOLD:
            ir_count += 1

ir_fraction = ir_count / len(sample_jpgs)
print(f"IR/night images (saturation < {IR_SATURATION_THRESHOLD}):")
print(f"  {ir_count} / {len(sample_jpgs)} sampled  ({ir_fraction:.0%})")
print()
if ir_fraction > 0.3:
    print(
        "WARNING: >30% of images appear to be IR/night — consider adding is_night metadata column."
    )
else:
    print("OK: IR fraction is manageable.")

## 7 — Image resolution distribution

In [ ]:
csv_path = SER_DIR / "metadata.csv"
if csv_path.exists():
    widths, heights = [], []
    with csv_path.open() as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                widths.append(int(row["width"]))
                heights.append(int(row["height"]))
            except (ValueError, KeyError):
                pass

    longer_sides = [max(w, h) for w, h in zip(widths, heights, strict=False)]
    print("Resolution summary (longer side):")
    print(f"  min:    {min(longer_sides)}")
    print(f"  median: {int(np.median(longer_sides))}")
    print(f"  max:    {max(longer_sides)}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].hist(longer_sides, bins=30, color="steelblue", edgecolor="white")
    axes[0].set_title("Longer side (pixels)")
    axes[0].set_xlabel("pixels")

    aspect_ratios = [w / h for w, h in zip(widths, heights, strict=False) if h > 0]
    axes[1].hist(aspect_ratios, bins=30, color="coral", edgecolor="white")
    axes[1].set_title("Aspect ratio (w/h)")
    axes[1].set_xlabel("w / h")
    plt.tight_layout()
    plt.show()
else:
    print("metadata.csv not found — run step 04 first.")

## 8 — Summary

In [ ]:
print("=" * 50)
print("QUALITY CHECK SUMMARY")
print("=" * 50)
print(f"  Total images (ser/):        {grand}")
print(f"  Total cropped (ser_cropped): {sum(1 for _ in CROPPED_DIR.glob('**/*.jpg'))}")
print(f"  Corrupt files:              {len(corrupt_ser) + len(corrupt_cropped)}")
print(f"  Classes:                    {len(all_labels)}")
print(f"  Class balance (max/min):    {max_count / min_count:.1f}x")
print(f"  IR/night fraction (sample): {ir_fraction:.0%}")
print()
ok = len(corrupt_ser) == 0 and len(corrupt_cropped) == 0
print("PASS" if ok else "FAIL (see warnings above)")